# Quantitative Multi-Asset Screener

This notebook builds a quantitative screener for the Reto Actinver investment universe.

The methodology:

1. Loads historical data from PostgreSQL.
2. Uses adjusted prices to calculate returns and momentum.
3. Applies freshness, data-quality and liquidity filters.
4. Standardizes factors within each asset category.
5. Computes a baseline multi-factor score.
6. Ranks instruments within STOCK, ETF and FUND categories.
7. Selects the top percentile as the candidate pool.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sqlalchemy import text


PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "database.py").is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project folder.")


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


from src.database import get_engine
from src.features import calculate_features
from src.screener import (
    build_universe,
    select_candidates
)

In [2]:
engine = get_engine()

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Database connection successful.")

Database connection successful.


## 1. Load market data

The screener uses `adjusted_close` for return-based calculations in order to reduce distortions caused by stock splits and other corporate actions.

`close` is retained as the observed market closing price.

Run the updated notebook 01 first to create the maintenance table and recover missing
adjusted history. Assets with invalid observations in the last 252 rows, or unresolved rejected
dates, are temporarily excluded. The pipeline's final reports explain what needs recovery.
We retain missing observations and use `pct_change(fill_method=None)` so a gap cannot
silently become a zero return.

In [3]:
prices = pd.read_sql(
    """
    SELECT
        p.date,
        p.asset_id,
        a.actinver_symbol,
        a.asset_type,
        p.close,
        p.adjusted_close,
        p.volume,
        COALESCE(r.pending_rejections, 0) AS pending_rejections
    FROM prices p
    JOIN assets a
        ON p.asset_id = a.asset_id
    LEFT JOIN (
        SELECT asset_id, COUNT(*) AS pending_rejections
        FROM public.price_rejections
        GROUP BY asset_id
    ) r ON r.asset_id = p.asset_id
    ORDER BY p.asset_id, p.date;
    """,
    engine
)

prices["date"] = pd.to_datetime(prices["date"])

engine.dispose()
if prices.empty:
    raise ValueError("Run notebook 01 first: no prices are available.")


In [4]:
prices = calculate_features(prices)

In [5]:
evaluation_date = (
    prices["date"].max()
)

universe = build_universe(
    prices=prices,
    evaluation_date=evaluation_date,
    momentum_weight=1/3,
    volatility_weight=1/3,
    drawdown_weight=1/3
)

candidate_pool = select_candidates(
    universe,
    cutoff=0.90
)

In [6]:
ranking = (
    universe
    .sort_values(
        [
            "asset_type",
            "category_rank"
        ]
    )
    .copy()
)

In [7]:
display(
    candidate_pool[
        [
            "actinver_symbol",
            "asset_type",
            "momentum_5d",
            "momentum_20d",
            "momentum_60d",
            "volatility_20d_ann",
            "current_drawdown",
            "score",
            "category_rank",
            "category_percentile"
        ]
    ]
)

,actinver_symbol,asset_type,momentum_5d,momentum_20d,momentum_60d,volatility_20d_ann,current_drawdown,score,category_rank,category_percentile
95561,DIABLOI10,ETF,0.002791,0.007477,0.052734,0.110564,-0.016423,0.817956,1.0,1.000000
13167,ACTI500B,FUND,0.027773,0.076730,0.066104,0.095876,0.000000,0.335258,1.0,1.000000
27839,ACTIRVTB,FUND,0.017668,0.044886,0.036936,0.047018,0.000000,0.288861,2.0,0.937500
203570,MSFT,STOCK,0.042470,0.129385,0.416774,0.206577,-0.047499,0.852737,1.0,1.000000
291906,TSMN,STOCK,0.083857,0.214685,0.154736,0.267855,0.000000,0.798579,2.0,0.972222
216490,NVDA,STOCK,0.057555,0.113517,0.181130,0.269722,0.000000,0.681280,3.0,0.944444
171355,KO,STOCK,0.002321,0.047131,0.066967,0.159812,-0.010993,0.631508,4.0,0.916667


In [8]:
candidate_pool[
    "asset_type"
].value_counts()

asset_type
STOCK    4
FUND     2
ETF      1
Name: count, dtype: int64

In [9]:
print(
    f"Evaluation date: "
    f"{evaluation_date.date()}"
)

print(
    f"Eligible assets: "
    f"{len(universe)}"
)

print(
    f"Candidate assets: "
    f"{len(candidate_pool)}"
)

Evaluation date: 2026-10-05
Eligible assets: 55
Candidate assets: 7


In [10]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

candidate_pool.to_csv(
    OUTPUT_DIR / "candidate_pool.csv",
    index=False
)

ranking.to_csv(
    OUTPUT_DIR / "screener_ranking.csv",
    index=False
)

print(
    f"Candidate pool saved to: "
    f"{OUTPUT_DIR / 'candidate_pool.csv'}"
)

Candidate pool saved to: c:\Users\alexi\OneDrive\Desktop\ProyectosPRO\bmv-quant-portfolio-system\data\processed\candidate_pool.csv
